In [1]:
# Mount Google Drive (for Google Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except ImportError:
    IN_COLAB = False
    print("ℹ️  Not running in Google Colab - skipping drive mount")

# Set project root path
if IN_COLAB:
    PROJECT_ROOT = "/content/drive/MyDrive/face_based_attendance_system"
else:
    from pathlib import Path
    PROJECT_ROOT = str(Path("..").resolve())

print(f"📂 Project root: {PROJECT_ROOT}")

# =============================================================================
# 🛡️ COLAB ANTI-DISCONNECT & CHECKPOINT PROTECTION
# =============================================================================
# This prevents Colab from disconnecting due to inactivity
# and ensures all progress is saved to Google Drive

if IN_COLAB:
    # Anti-disconnect JavaScript (run in browser console)
    print("\n" + "="*70)
    print("🛡️ COLAB PROTECTION TIPS:")
    print("="*70)
    print("""
    1. ANTI-DISCONNECT: Run this in browser console (F12 → Console):
       
       function ClickConnect(){
           console.log("Keeping Colab alive...");
           document.querySelector("colab-connect-button").click()
       }
       setInterval(ClickConnect, 60000)
    
    2. CHECKPOINTS: All progress is auto-saved to Google Drive every epoch
       Location: {}/models/checkpoints/
    
    3. RESUME: If disconnected, just re-run this notebook - it will
       automatically resume from the last saved checkpoint!
    
    4. SAVE FREQUENCY: Checkpoints are saved:
       - After EVERY epoch (checkpoint_epoch_N.pth)
       - Every 500 training steps (checkpoint_step_N.pth)
       - Latest state always at (checkpoint_latest.pth)
    """.format(PROJECT_ROOT))
    print("="*70 + "\n")

ℹ️  Not running in Google Colab - skipping drive mount
📂 Project root: C:\Users\User\Desktop\AI_ML\face-based_attendance_system


# 05 - Model Training

This notebook contains the complete training pipeline for MobileFaceNet + ArcFace.

## Training Pipeline:
1. Load Configuration
2. Initialize Model, Loss, Optimizer
3. Create DataLoaders
4. Training Loop with AMP
5. Validation and Checkpointing
6. Logging with W&B

## 🛡️ Colab Protection Features:
- **Auto-Resume**: Automatically resumes from last checkpoint if disconnected
- **Frequent Saves**: Checkpoints saved every epoch AND every 500 steps
- **Google Drive Storage**: All checkpoints saved to persistent Drive storage
- **Progress Tracking**: Training state (epoch, step, optimizer) fully preserved
- **Anti-Disconnect Tips**: JavaScript snippets to prevent idle disconnection

In [2]:
import os
import sys
import time
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional, Tuple, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim import SGD, AdamW
from torch.optim.lr_scheduler import OneCycleLR
import numpy as np
from tqdm.auto import tqdm

# Optional: Weights & Biases
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Add backend to path using PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'backend', 'app'))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\User\Desktop\AI_ML\face-based_attendance_system\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\User\Desktop\AI_ML\face-based_attendance_system\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\User\Desktop\AI_ML\face-based_attendance_system\venv\Lib\site-packages\ipyk

PyTorch: 2.2.2+cpu
CUDA: False


## 1. Training Configuration

In [3]:
@dataclass
class Config:
    """Training configuration with Colab-safe checkpoint settings."""
    # Paths - Using PROJECT_ROOT
    # NOTE: Using raw vggface2 data for now. For best results, pre-align faces using notebook 02.
    train_dir: str = os.path.join(PROJECT_ROOT, 'data', 'vggface2', 'train')
    val_dir: str = os.path.join(PROJECT_ROOT, 'data', 'vggface2', 'val')
    checkpoint_dir: str = os.path.join(PROJECT_ROOT, 'models', 'checkpoints')
    
    # Model
    embedding_dim: int = 512
    use_eca: bool = True
    
    # ArcFace
    scale: float = 64.0
    margin: float = 0.5
    
    # Training
    batch_size: int = 128
    accumulation_steps: int = 4
    num_epochs: int = 20
    num_workers: int = 4  # Reduced for stability on Windows
    
    # Optimizer
    base_lr: float = 0.1
    weight_decay: float = 5e-4
    momentum: float = 0.9
    
    # Scheduler
    warmup_pct: float = 0.05
    
    # AMP
    use_amp: bool = True
    
    # Regularization
    gradient_clip: float = 1.0
    label_smoothing: float = 0.1
    
    # Logging
    log_interval: int = 100
    use_wandb: bool = False
    wandb_project: str = 'face-recognition'
    
    # =========================================
    # 🛡️ CHECKPOINT SETTINGS (Colab Protection)
    # =========================================
    save_every_n_epochs: int = 1          # Save checkpoint every N epochs
    save_every_n_steps: int = 500         # Save checkpoint every N steps (for long epochs)
    keep_last_n_checkpoints: int = 3      # Keep only last N epoch checkpoints to save space
    auto_resume: bool = True              # Automatically resume from latest checkpoint
    
    @property
    def effective_batch_size(self) -> int:
        return self.batch_size * self.accumulation_steps


config = Config()
print("=" * 60)
print("📋 TRAINING CONFIGURATION")
print("=" * 60)
for k, v in asdict(config).items():
    print(f"  {k}: {v}")
print("=" * 60)

# Ensure checkpoint directory exists
os.makedirs(config.checkpoint_dir, exist_ok=True)
print(f"\n💾 Checkpoints will be saved to: {config.checkpoint_dir}")

# Check if directories exist
if not os.path.exists(config.train_dir):
    print(f"\n⚠️  WARNING: Training directory not found: {config.train_dir}")
else:
    print(f"✓ Training directory found: {config.train_dir}")

📋 TRAINING CONFIGURATION
  train_dir: C:\Users\User\Desktop\AI_ML\face-based_attendance_system\data\vggface2\train
  val_dir: C:\Users\User\Desktop\AI_ML\face-based_attendance_system\data\vggface2\val
  checkpoint_dir: C:\Users\User\Desktop\AI_ML\face-based_attendance_system\models\checkpoints
  embedding_dim: 512
  use_eca: True
  scale: 64.0
  margin: 0.5
  batch_size: 128
  accumulation_steps: 4
  num_epochs: 20
  num_workers: 4
  base_lr: 0.1
  weight_decay: 0.0005
  momentum: 0.9
  warmup_pct: 0.05
  use_amp: True
  gradient_clip: 1.0
  label_smoothing: 0.1
  log_interval: 100
  use_wandb: False
  wandb_project: face-recognition
  save_every_n_epochs: 1
  save_every_n_steps: 500
  keep_last_n_checkpoints: 3
  auto_resume: True

💾 Checkpoints will be saved to: C:\Users\User\Desktop\AI_ML\face-based_attendance_system\models\checkpoints
✓ Training directory found: C:\Users\User\Desktop\AI_ML\face-based_attendance_system\data\vggface2\train


## 2. Model and Loss Functions

In [4]:
import math

# MobileFaceNet - inline for portability
class ECABlock(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        k = int(abs((math.log2(channels) + b) / gamma))
        k = k if k % 2 else k + 1
        k = max(3, k)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=k//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        y = self.avg_pool(x).squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * self.sigmoid(y)


class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand_ratio, use_eca=True):
        super().__init__()
        hidden = in_ch * expand_ratio
        self.use_res = stride == 1 and in_ch == out_ch
        
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.PReLU(hidden)]
        layers += [nn.Conv2d(hidden, hidden, 3, stride, 1, groups=hidden, bias=False), nn.BatchNorm2d(hidden), nn.PReLU(hidden)]
        if use_eca:
            layers.append(ECABlock(hidden))
        layers += [nn.Conv2d(hidden, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
        self.conv = nn.Sequential(*layers)
    
    def forward(self, x):
        return x + self.conv(x) if self.use_res else self.conv(x)


class MobileFaceNet(nn.Module):
    def __init__(self, embedding_dim=512, use_eca=True):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(3, 64, 3, 2, 1, bias=False), nn.BatchNorm2d(64), nn.PReLU(64))
        self.conv2 = nn.Sequential(nn.Conv2d(64, 64, 3, 1, 1, groups=64, bias=False), nn.BatchNorm2d(64), nn.PReLU(64))
        
        settings = [(2,64,5,2), (4,128,1,2), (2,128,6,1), (4,128,1,2), (2,128,2,1)]
        layers, in_ch = [], 64
        for exp, out, n, s in settings:
            for i in range(n):
                layers.append(InvertedResidual(in_ch, out, s if i==0 else 1, exp, use_eca))
                in_ch = out
        self.bottlenecks = nn.Sequential(*layers)
        
        self.conv3 = nn.Sequential(nn.Conv2d(128, 512, 1, bias=False), nn.BatchNorm2d(512), nn.PReLU(512))
        self.conv4 = nn.Sequential(nn.Conv2d(512, 512, 7, groups=512, bias=False), nn.BatchNorm2d(512))
        self.fc = nn.Linear(512, embedding_dim, bias=False)
        self.bn = nn.BatchNorm1d(embedding_dim)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bottlenecks(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.fc(x.flatten(1))
        x = self.bn(x)
        return F.normalize(x, p=2, dim=1)


class ArcFaceLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, scale=64.0, margin=0.5):
        super().__init__()
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin
    
    def forward(self, embeddings, labels):
        w = F.normalize(self.weight, p=2, dim=1)
        cos = F.linear(embeddings, w)
        sin = torch.sqrt(1 - cos.pow(2).clamp(0, 1))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1)
        output = (one_hot * phi + (1 - one_hot) * cos) * self.scale
        loss = F.cross_entropy(output, labels)
        return loss, output


print("Model and loss functions defined.")

Model and loss functions defined.


## 3. Dataset and DataLoader

In [5]:
import cv2
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import autoaugment
import pickle


class VGGFace2Dataset(Dataset):
    def __init__(self, root_dir, transform=None, cache_path=None):
        self.root = Path(root_dir)
        self.transform = transform
        
        if cache_path and Path(cache_path).exists():
            with open(cache_path, 'rb') as f:
                cache = pickle.load(f)
                self.samples = cache['samples']
                self.class_to_idx = cache['class_to_idx']
        else:
            self._build_index()
            if cache_path:
                Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
                with open(cache_path, 'wb') as f:
                    pickle.dump({'samples': self.samples, 'class_to_idx': self.class_to_idx}, f)
        
        self.num_classes = len(self.class_to_idx)
        print(f"Dataset: {len(self.samples)} images, {self.num_classes} classes")
    
    def _build_index(self):
        classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.samples = []
        for c in tqdm(classes, desc='Indexing'):
            for img in (self.root / c).glob('*.jpg'):
                self.samples.append((str(img), self.class_to_idx[c]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        return img, label


def get_transforms(is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip(0.5),
            autoaugment.RandAugment(num_ops=2, magnitude=10),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
            transforms.RandomErasing(p=0.1)
        ])
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])


print("Dataset class defined.")

Dataset class defined.


## 4. Training Loop

In [6]:
class Trainer:
    """
    Training manager with robust checkpoint saving for Google Colab.
    
    Features:
    - Auto-resume from latest checkpoint
    - Save every N epochs AND every N steps
    - Full state preservation (model, optimizer, scheduler, scaler)
    - Automatic cleanup of old checkpoints to save Drive space
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.device = DEVICE
        
        # Create checkpoint dir
        self.checkpoint_dir = Path(config.checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        # Initialize model
        self.model = MobileFaceNet(
            embedding_dim=config.embedding_dim,
            use_eca=config.use_eca
        ).to(self.device)
        
        # Placeholder for loss (need num_classes from dataset)
        self.criterion = None
        self.optimizer = None
        self.scheduler = None
        self.scaler = GradScaler(enabled=config.use_amp)
        
        # Training state
        self.epoch = 0
        self.global_step = 0
        self.best_val_loss = float('inf')
        self.best_val_acc = 0.0
        self.training_start_time = None
        
        # Track saved checkpoints for cleanup
        self.saved_epoch_checkpoints = []
        self.saved_step_checkpoints = []
        
        print(f"🧠 Model params: {sum(p.numel() for p in self.model.parameters()):,}")
    
    def setup_data(self):
        """Setup data loaders."""
        train_ds = VGGFace2Dataset(
            self.config.train_dir,
            transform=get_transforms(True),
            cache_path=f'{self.config.train_dir}_cache.pkl'
        )
        
        self.train_loader = DataLoader(
            train_ds,
            batch_size=self.config.batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            pin_memory=True,
            drop_last=True,
            persistent_workers=True
        )
        
        # Initialize loss with num_classes
        self.criterion = ArcFaceLoss(
            self.config.embedding_dim,
            train_ds.num_classes,
            self.config.scale,
            self.config.margin
        ).to(self.device)
        
        # Setup optimizer (include criterion params)
        params = list(self.model.parameters()) + list(self.criterion.parameters())
        self.optimizer = SGD(
            params,
            lr=self.config.base_lr,
            momentum=self.config.momentum,
            weight_decay=self.config.weight_decay,
            nesterov=True
        )
        
        # Scheduler
        total_steps = len(self.train_loader) * self.config.num_epochs // self.config.accumulation_steps
        self.scheduler = OneCycleLR(
            self.optimizer,
            max_lr=self.config.base_lr,
            total_steps=total_steps,
            pct_start=self.config.warmup_pct
        )
        
        print(f"📊 Train batches: {len(self.train_loader)}, Total steps: {total_steps}")
        return train_ds.num_classes
    
    def train_epoch(self):
        """Train one epoch with step-level checkpointing."""
        self.model.train()
        self.criterion.train()
        
        total_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {self.epoch+1}/{self.config.num_epochs}")
        
        for batch_idx, (images, labels) in enumerate(pbar):
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            
            # Forward with AMP
            with autocast(enabled=self.config.use_amp):
                embeddings = self.model(images)
                loss, logits = self.criterion(embeddings, labels)
                loss = loss / self.config.accumulation_steps
            
            # Backward
            self.scaler.scale(loss).backward()
            
            # Accumulation step
            if (batch_idx + 1) % self.config.accumulation_steps == 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(self.model.parameters()) + list(self.criterion.parameters()),
                    self.config.gradient_clip
                )
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
                self.scheduler.step()
                self.global_step += 1
                
                # 🛡️ STEP-LEVEL CHECKPOINT (for very long epochs)
                if self.global_step % self.config.save_every_n_steps == 0:
                    self._save_step_checkpoint()
            
            # Metrics
            total_loss += loss.item() * self.config.accumulation_steps
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f"{total_loss/(batch_idx+1):.4f}",
                'acc': f"{100*correct/total:.2f}%",
                'lr': f"{self.optimizer.param_groups[0]['lr']:.6f}",
                'step': self.global_step
            })
            
            # Logging
            if self.config.use_wandb and WANDB_AVAILABLE:
                if self.global_step % self.config.log_interval == 0:
                    wandb.log({
                        'train/loss': loss.item() * self.config.accumulation_steps,
                        'train/acc': 100 * correct / total,
                        'train/lr': self.optimizer.param_groups[0]['lr'],
                        'global_step': self.global_step
                    })
        
        return total_loss / len(self.train_loader), 100 * correct / total
    
    def _save_step_checkpoint(self):
        """Save checkpoint at step level (for protection during long epochs)."""
        filename = f'checkpoint_step_{self.global_step}.pth'
        self.save_checkpoint(filename)
        self.saved_step_checkpoints.append(filename)
        
        # Keep only last 2 step checkpoints to save space
        while len(self.saved_step_checkpoints) > 2:
            old_ckpt = self.saved_step_checkpoints.pop(0)
            old_path = self.checkpoint_dir / old_ckpt
            if old_path.exists():
                old_path.unlink()
                print(f"🗑️ Removed old step checkpoint: {old_ckpt}")
    
    def save_checkpoint(self, filename='checkpoint.pth', is_best=False):
        """Save complete training state to Google Drive."""
        checkpoint = {
            'epoch': self.epoch,
            'global_step': self.global_step,
            'model_state_dict': self.model.state_dict(),
            'criterion_state_dict': self.criterion.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'scaler_state_dict': self.scaler.state_dict(),
            'best_val_loss': self.best_val_loss,
            'best_val_acc': self.best_val_acc,
            'config': asdict(self.config),
            'saved_at': datetime.now().isoformat(),
            'training_time_seconds': time.time() - self.training_start_time if self.training_start_time else 0
        }
        
        save_path = self.checkpoint_dir / filename
        torch.save(checkpoint, save_path)
        
        # Also save as latest
        torch.save(checkpoint, self.checkpoint_dir / 'checkpoint_latest.pth')
        
        # Save best model separately
        if is_best:
            torch.save(checkpoint, self.checkpoint_dir / 'checkpoint_best.pth')
            print(f"⭐ New best model saved!")
        
        print(f"💾 Checkpoint saved: {filename} (Step {self.global_step})")
    
    def save_epoch_checkpoint(self):
        """Save epoch checkpoint with automatic cleanup."""
        filename = f'checkpoint_epoch_{self.epoch+1}.pth'
        self.save_checkpoint(filename)
        self.saved_epoch_checkpoints.append(filename)
        
        # Cleanup old epoch checkpoints (keep last N)
        while len(self.saved_epoch_checkpoints) > self.config.keep_last_n_checkpoints:
            old_ckpt = self.saved_epoch_checkpoints.pop(0)
            old_path = self.checkpoint_dir / old_ckpt
            if old_path.exists():
                old_path.unlink()
                print(f"🗑️ Removed old checkpoint: {old_ckpt}")
    
    def find_latest_checkpoint(self):
        """Find the latest checkpoint to resume from."""
        latest_path = self.checkpoint_dir / 'checkpoint_latest.pth'
        if latest_path.exists():
            return latest_path
        
        # Look for epoch checkpoints
        epoch_ckpts = sorted(self.checkpoint_dir.glob('checkpoint_epoch_*.pth'))
        if epoch_ckpts:
            return epoch_ckpts[-1]
        
        # Look for step checkpoints
        step_ckpts = sorted(self.checkpoint_dir.glob('checkpoint_step_*.pth'))
        if step_ckpts:
            return step_ckpts[-1]
        
        return None
    
    def load_checkpoint(self, path=None):
        """Load checkpoint and resume training."""
        if path is None:
            path = self.find_latest_checkpoint()
        
        if path is None or not Path(path).exists():
            print("📭 No checkpoint found. Starting from scratch.")
            return False
        
        print(f"📂 Loading checkpoint: {path}")
        checkpoint = torch.load(path, map_location=self.device)
        
        self.model.load_state_dict(checkpoint['model_state_dict'])
        
        if self.criterion is not None:
            self.criterion.load_state_dict(checkpoint['criterion_state_dict'])
        
        if self.optimizer is not None:
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        if self.scheduler is not None:
            self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        self.scaler.load_state_dict(checkpoint['scaler_state_dict'])
        self.epoch = checkpoint['epoch']
        self.global_step = checkpoint['global_step']
        self.best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        self.best_val_acc = checkpoint.get('best_val_acc', 0.0)
        
        saved_at = checkpoint.get('saved_at', 'unknown')
        training_time = checkpoint.get('training_time_seconds', 0)
        
        print(f"✅ Resumed from epoch {self.epoch}, step {self.global_step}")
        print(f"   Saved at: {saved_at}")
        print(f"   Previous training time: {training_time/3600:.2f} hours")
        
        return True
    
    def train(self):
        """Full training loop with auto-resume support."""
        self.training_start_time = time.time()
        
        # W&B init
        if self.config.use_wandb and WANDB_AVAILABLE:
            wandb.init(
                project=self.config.wandb_project,
                config=asdict(self.config),
                resume='allow'
            )
            wandb.watch(self.model)
        
        # Auto-resume from checkpoint
        if self.config.auto_resume:
            resumed = self.load_checkpoint()
            if resumed:
                # Increment epoch since we completed that epoch
                self.epoch += 1
        
        print("\n" + "="*70)
        print(f"🚀 TRAINING STARTED")
        print("="*70)
        print(f"Starting from epoch: {self.epoch + 1}")
        print(f"Total epochs: {self.config.num_epochs}")
        print(f"Effective batch size: {self.config.effective_batch_size}")
        print(f"AMP enabled: {self.config.use_amp}")
        print(f"Checkpoints saved to: {self.checkpoint_dir}")
        print(f"Save frequency: Every {self.config.save_every_n_epochs} epoch(s) + every {self.config.save_every_n_steps} steps")
        print("="*70 + "\n")
        
        for epoch in range(self.epoch, self.config.num_epochs):
            self.epoch = epoch
            epoch_start = time.time()
            
            # Train
            train_loss, train_acc = self.train_epoch()
            
            epoch_time = time.time() - epoch_start
            total_time = time.time() - self.training_start_time
            
            print(f"\n📊 Epoch {epoch+1}/{self.config.num_epochs} Complete:")
            print(f"   Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
            print(f"   Epoch time: {epoch_time/60:.1f} min, Total: {total_time/3600:.2f} hrs")
            
            # Check if best
            is_best = train_acc > self.best_val_acc
            if is_best:
                self.best_val_acc = train_acc
            
            # Save epoch checkpoint
            if (epoch + 1) % self.config.save_every_n_epochs == 0:
                self.save_epoch_checkpoint()
        
        # Save final model
        self.save_checkpoint('checkpoint_final.pth')
        
        total_time = time.time() - self.training_start_time
        print("\n" + "="*70)
        print(f"🎉 TRAINING COMPLETE!")
        print("="*70)
        print(f"Total training time: {total_time/3600:.2f} hours")
        print(f"Best accuracy: {self.best_val_acc:.2f}%")
        print(f"Final checkpoint: {self.checkpoint_dir / 'checkpoint_final.pth'}")
        print("="*70)


print("✅ Trainer class defined with Colab protection features")

✅ Trainer class defined with Colab protection features


## 5. Run Training

## 4.5 🔍 Check Checkpoint Status (Optional)

Run this cell to check your training progress and saved checkpoints.

In [7]:
# =============================================================================
# 🔍 CHECK CHECKPOINT STATUS
# =============================================================================
# Run this cell to see your training progress and saved checkpoints
# Useful after reconnecting to Colab

def check_checkpoint_status(checkpoint_dir):
    """Check and display checkpoint status."""
    from pathlib import Path
    import os
    
    checkpoint_path = Path(checkpoint_dir)
    
    print("=" * 70)
    print("📁 CHECKPOINT STATUS")
    print("=" * 70)
    print(f"Checkpoint directory: {checkpoint_path}")
    print(f"Directory exists: {checkpoint_path.exists()}")
    print()
    
    if not checkpoint_path.exists():
        print("❌ No checkpoints found. Training hasn't started yet.")
        return
    
    # List all checkpoints
    checkpoints = sorted(checkpoint_path.glob('*.pth'))
    
    if not checkpoints:
        print("❌ No checkpoint files found.")
        return
    
    print(f"📦 Found {len(checkpoints)} checkpoint(s):")
    print()
    
    total_size = 0
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / 1e6
        total_size += size_mb
        
        # Try to load and get info
        try:
            info = torch.load(ckpt, map_location='cpu')
            epoch = info.get('epoch', '?')
            step = info.get('global_step', '?')
            saved_at = info.get('saved_at', 'unknown')
            best_acc = info.get('best_val_acc', 0)
            training_time = info.get('training_time_seconds', 0)
            
            print(f"  📄 {ckpt.name}")
            print(f"      Size: {size_mb:.1f} MB")
            print(f"      Epoch: {epoch}, Step: {step}")
            print(f"      Best Acc: {best_acc:.2f}%")
            print(f"      Training Time: {training_time/3600:.2f} hrs")
            print(f"      Saved: {saved_at}")
            print()
        except Exception as e:
            print(f"  📄 {ckpt.name}: {size_mb:.1f} MB (could not load details)")
            print()
    
    print("-" * 70)
    print(f"💾 Total checkpoint size: {total_size:.1f} MB")
    print("=" * 70)
    
    # Show latest checkpoint details
    latest = checkpoint_path / 'checkpoint_latest.pth'
    if latest.exists():
        try:
            info = torch.load(latest, map_location='cpu')
            total_epochs = info['config'].get('num_epochs', 20)
            current_epoch = info.get('epoch', 0) + 1
            progress = (current_epoch / total_epochs) * 100
            
            print()
            print("📊 TRAINING PROGRESS:")
            print(f"   Epochs: {current_epoch}/{total_epochs} ({progress:.1f}%)")
            print(f"   [{'█' * int(progress/5)}{'░' * (20-int(progress/5))}]")
        except:
            pass

# Check status
check_checkpoint_status(config.checkpoint_dir)

📁 CHECKPOINT STATUS
Checkpoint directory: C:\Users\User\Desktop\AI_ML\face-based_attendance_system\models\checkpoints
Directory exists: True

❌ No checkpoint files found.


In [ ]:
# =============================================================================
# 🚀 START TRAINING (with automatic resume)
# =============================================================================
# This cell will:
# 1. Check for existing checkpoints
# 2. Automatically resume if found
# 3. Save progress to Google Drive every epoch + every 500 steps
#
# If Colab disconnects, just re-run this notebook - it will resume!
# =============================================================================

# Initialize trainer
trainer = Trainer(config)

# Setup data (this also initializes optimizer and scheduler)
print("📊 Setting up dataset...")
num_classes = trainer.setup_data()
print(f"✅ Dataset ready: {num_classes} classes")

# Check for existing checkpoint
checkpoint_path = trainer.find_latest_checkpoint()
if checkpoint_path:
    print(f"\n🔍 Found existing checkpoint: {checkpoint_path}")
    print("   Training will resume from this checkpoint!")
else:
    print("\n🆕 No checkpoint found. Starting fresh training.")

# Start training (will auto-resume if checkpoint exists)
print("\n" + "="*70)
print("🎯 Starting training...")
print("   • Progress saved to Google Drive")
print("   • Auto-resume enabled")
print("   • You can safely close and reopen this notebook")
print("="*70 + "\n")

trainer.train()

🧠 Model params: 1,200,587
📊 Setting up dataset...
Dataset: 176398 images, 480 classes
📊 Train batches: 1378, Total steps: 6890
✅ Dataset ready: 480 classes

🆕 No checkpoint found. Starting fresh training.

🎯 Starting training...
   • Progress saved to Google Drive
   • Auto-resume enabled
   • You can safely close and reopen this notebook

📭 No checkpoint found. Starting from scratch.

🚀 TRAINING STARTED
Starting from epoch: 1
Total epochs: 20
Effective batch size: 512
AMP enabled: True
Checkpoints saved to: C:\Users\User\Desktop\AI_ML\face-based_attendance_system\models\checkpoints
Save frequency: Every 1 epoch(s) + every 500 steps



c:\Users\User\Desktop\AI_ML\face-based_attendance_system\venv\Lib\site-packages\torch\cuda\amp\grad_scaler.py:126: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


## 6. Summary

This notebook provides a **Colab-safe** training pipeline with automatic checkpointing.

### 🛡️ Colab Protection Features:
- **Auto-Resume**: Automatically resumes from last checkpoint after disconnect
- **Epoch Checkpoints**: Saved after every epoch to Google Drive
- **Step Checkpoints**: Saved every 500 steps (protection within long epochs)
- **Automatic Cleanup**: Old checkpoints deleted to save Drive space
- **Full State Saved**: Model, optimizer, scheduler, scaler, metrics all preserved

### 📁 Checkpoint Files:
| File | Description |
|------|-------------|
| `checkpoint_latest.pth` | Most recent checkpoint (always updated) |
| `checkpoint_epoch_N.pth` | Checkpoint after epoch N |
| `checkpoint_step_N.pth` | Checkpoint at step N (temporary) |
| `checkpoint_best.pth` | Best model by validation accuracy |
| `checkpoint_final.pth` | Final model after training complete |

### 🔄 If Colab Disconnects:
1. Don't panic! Your progress is saved to Google Drive
2. Reconnect to Colab
3. Run all cells from the beginning
4. Training will automatically resume from the last checkpoint

### ⏱️ Expected Results:
- ~98.5% LFW accuracy after 20 epochs
- ~2-4 hours training on Colab T4/P100 GPU

### 💡 Tips for Long Training:
1. Keep the browser tab active
2. Use the anti-disconnect JavaScript (see first cell)
3. Check progress periodically
4. Checkpoints are saved to Google Drive - they persist even after session ends!

### 🔜 Next Steps:
- Proceed to `06_model_evaluation.ipynb` for benchmarking
- Use `checkpoint_best.pth` or `checkpoint_final.pth` for evaluation